In [1]:
import pandas as pd
from thefuzz import process, fuzz
import os
import json
import time
import requests

import cv2  # OpenCV
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import entropy


In [30]:
# Load the datasets from the 'data' folder
# Note: Ensure the filenames match exactly what you have in your folder
tapas_df = pd.read_csv('data/tapas_data.csv') 


# Displaying the structure of the data to identify key columns
print("Tapas Dataset Columns:")
print(tapas_df.columns)

# Preview the first few rows
tapas_df.head()

Tapas Dataset Columns:
Index(['title', 'item_id', 'link', 'cover', 'creators', 'genres', 'views',
       'subscribers', 'likes', 'banner', 'details', 'tags', 'episodes',
       'released'],
      dtype='str')


,title,item_id,link,cover,creators,genres,views,subscribers,likes,banner,details,tags,episodes,released
0,My Gentle Giant,140273,https://tapas.io/series/My-Gentle-Giant/info,https://d30womf5coomej.cloudfront.net/sa/55/a7...,['EmAuthor'],"['BL', 'LGBTQ+', 'Slice of life']","29,109,978 views","241,427 subscribers","2,934,170 likes",https://d30womf5coomej.cloudfront.net/sa/01/c4...,Jun Watanabe is your average outcast. He's a t...,"['#gay', '#soft', '#comedy', '#Angst', '#bl', ...",176,"Sep 28, 2020"
1,DaiMaou,36492,https://tapas.io/series/daimaou/info,https://d30womf5coomej.cloudfront.net/sa/ee/47...,['Amanduur'],"['BL', 'Comedy', 'Fantasy']","24,186,270 views","103,218 subscribers","2,453,503 likes",https://d30womf5coomej.cloudfront.net/sa/e9/5b...,TL;DR: Shitty comedy masquerading as an actual...,"['#gay', '#Fantasy', '#romance', '#comedy', '#...",479,"Jul 08, 2020"
2,Idiots Don't Catch Colds,67447,https://tapas.io/series/Idiots-Dont-Catch-Cold...,https://d30womf5coomej.cloudfront.net/sa/6e/68...,['Aina Palm'],['BL'],"18,534,129 views","132,648 subscribers","1,851,428 likes",https://d30womf5coomej.cloudfront.net/sa/49/92...,There is only one guy Souta absolutely can't s...,"['#romance', '#drama', '#comedy', '#Soccer', '...",233,"Sep 02, 2020"
3,Jamie,110007,https://tapas.io/series/Jamie/info,https://d30womf5coomej.cloudfront.net/sa/a9/05...,"['Bre Indigo', 'Tami']","['LGBTQ+', 'Drama', 'Slice of life']","14,263,432 views","126,167 subscribers","1,452,529 likes",https://d30womf5coomej.cloudfront.net/sa/94/ce...,[ Coming of Age | LGBTQ+ | Young Adult ]\r\n\r...,"['#friendship', '#queer', '#crush', '#lgbt', '...",138,"Jun 04, 2020"
4,FANGS,155459,https://tapas.io/series/fangscomic/info,https://d30womf5coomej.cloudfront.net/sa/18/a9...,['Sarah Andersen'],"['Romance', 'Comedy']","36,237,626 views","160,308 subscribers","1,343,073 likes",6,Vamp is three hundred years old but in all tha...,[],78,"Oct 31, 2019"


In [31]:
webtoon_df = pd.read_csv('data/webtoon_data.csv')
print("\nWebtoons Dataset Columns:")
print(webtoon_df.columns)

webtoon_df.head()


Webtoons Dataset Columns:
Index(['webtoon_id', 'title', 'genre', 'thumbnail', 'summary', 'episodes',
       'Created by', 'view', 'subscribe', 'grade', 'released_date', 'url',
       'cover', 'likes', 'Written by', 'Art by', 'Adapted by',
       'Original work by', 'Assisted by'],
      dtype='str')


,webtoon_id,title,genre,thumbnail,summary,episodes,Created by,view,subscribe,grade,released_date,url,cover,likes,Written by,Art by,Adapted by,Original work by,Assisted by
0,1218,Let's Play,Romance,https://webtoon-phinf.pstatic.net/20210629_103...,"She’s young, single and about to achieve her d...",171,Leeanne M. Krecic (Mongie),606.5M,4.6M,9.59,"Nov 6, 2017",https://www.webtoons.com/en/romance/letsplay/l...,https://webtoon-phinf.pstatic.net/20210629_163...,37.2M,NaN,NaN,NaN,NaN,NaN
1,1436,True Beauty,Romance,https://webtoon-phinf.pstatic.net/20210129_175...,"After binge-watching beauty videos online, a s...",197,Yaongyi,874M,7M,9.53,"Aug 15, 2018",https://www.webtoons.com/en/romance/truebeauty...,https://webtoon-phinf.pstatic.net/20210129_65/...,46.2M,NaN,NaN,NaN,NaN,NaN
2,2135,The Remarried Empress,Fantasy,https://webtoon-phinf.pstatic.net/20200904_29/...,Navier Ellie Trovi was an empress perfect in e...,110,NaN,231.3M,2.5M,9.87,"Sep 5, 2020",https://www.webtoons.com/en/fantasy/the-remarr...,https://webtoon-phinf.pstatic.net/20200904_268...,21.2M,Alphatart,Sumpul,HereLee,NaN,NaN
3,1798,Midnight Poppy Land,Romance,https://webtoon-phinf.pstatic.net/20191119_132...,After making a grisly discovery in the country...,99,Lilydusk,198.8M,2.3M,9.80,"Nov 22, 2019",https://www.webtoons.com/en/romance/midnight-p...,https://webtoon-phinf.pstatic.net/20191119_163...,13.5M,NaN,NaN,NaN,NaN,NaN
4,3416,Reunion,Romance,https://webtoon-phinf.pstatic.net/20220311_196...,"After moving away for a decade, Rhea returns t...",9,stephattyy,7.1M,"629,872",9.77,"Mar 17, 2022",https://www.webtoons.com/en/romance/reunion/li...,https://webtoon-phinf.pstatic.net/20220311_14/...,"570,151",NaN,NaN,NaN,NaN,NaN


In [34]:

def create_master_list_with_stats(df_base, df_to_add, threshold=90):
    """
    Unified dataset creator that tracks matching statistics.
    """
    base_titles = df_base['title'].tolist()
    titles_to_check = df_to_add['title'].tolist()
    
    unique_entries = []
    same_titles_found = [] # To track duplicates/matches
    unique_titles_found = [] # To track new unique entries
    
    print(f"Comparing {len(titles_to_check)} titles against base of {len(base_titles)}...")
    
    for title in titles_to_check:
        # Fuzzy Matching logic to align titles 
        match, score = process.extractOne(title, base_titles, scorer=fuzz.token_sort_ratio)
        
        if score >= threshold:
            # This is a 'Same' title (Duplicate across platforms)
            same_titles_found.append({'original': title, 'matched_to': match, 'score': score})
        else:
            # This is a 'Unique' title
            new_row = df_to_add[df_to_add['title'] == title].iloc[0]
            unique_entries.append(new_row)
            unique_titles_found.append(title)
            
    # Combine to create the unified dataset 
    if unique_entries:
        additional_df = pd.DataFrame(unique_entries)
        master_df = pd.concat([df_base, additional_df], ignore_index=True)
    else:
        master_df = df_base.copy()

    # Final Stats Output
    print("-" * 30)
    print(f"MATCHING RESULTS:")
    print(f"Total 'Same' titles found (Duplicates): {len(same_titles_found)}")
    print(f"Total 'Unique' titles added: {len(unique_titles_found)}")
    print(f"Final Dataset Size: {len(master_df)}")
    print("-" * 30)
    
    # Returning the lists so you can inspect them if you want
    return master_df, same_titles_found

# Run the function
master_list, duplicates_log = create_master_list_with_stats(webtoon_df, tapas_df)

# If you want to see a few examples of what it thought was the 'same':
print("Sample of matches found:")
print(pd.DataFrame(duplicates_log).head(10))

Comparing 737 titles against base of 734...
------------------------------
MATCHING RESULTS:
Total 'Same' titles found (Duplicates): 2
Total 'Unique' titles added: 735
Final Dataset Size: 1469
------------------------------
Sample of matches found:
    original matched_to  score
0  Nevermore  Nevermore    100
1   Flawless   Flawless    100


> **In here**, Nevermore and Flawless are actualy not the same manhwa, they are different manhwas named same in different  but I decided to not included them in dataset because they may cause confusion.

### **Metadata Enrichment & API Integration**

#### **Purpose & Methodology**
The `smart_enrichment_loop` function serves as the **Data Enrichment** backbone of this project. Its primary goal is to bridge the gap between static Kaggle datasets (Tapas/Webtoon) and real-time performance metrics provided by the **Jikan API (MyAnimeList)**, such as global rank, popularity, and user scores.

#### **Key Technical Features**
* **Cache-First Architecture:** To ensure efficiency and minimize network overhead, the function implements a local caching system. It checks the `data/json_cache` directory for existing data before initiating any external API calls.
* **Fuzzy Name Matching:** To resolve naming discrepancies between different platforms (e.g., "True Beauty" vs. "True Beauty (2020)"), a **Levenshtein Distance** algorithm (`fuzz.token_sort_ratio`) is utilized. Metadata is only committed to the dataset if a confidence score of **>80%** is achieved.
* **Ethical Scraping & Rate Limiting:** The function is designed to respect Jikan’s server constraints. It includes a mandatory **1.2-second delay** between requests and an automated **30-second "cool-down" period** if a `429 Rate Limit` error is triggered.
* **Data Integrity:** This process ensures that every Manhwa in our final analysis has verified metadata, allowing us to correlate **Art Style Evolution** with commercial success.

> **Note on Execution Output:** The sequential fetching logs for the 722 titles have been cleared to maintain document readability and professional presentation.

In [ ]:

# Constants for the enrichment pipeline [cite: 9]
CACHE_DIR = 'data/json_cache'
os.makedirs(CACHE_DIR, exist_ok=True)

def get_sanitized_filename(title):
    """Creates a safe filename for the JSON cache."""
    return "".join(x for x in title if x.isalnum()).lower() + ".json"

def smart_enrichment_loop(master_df, limit=None):
    """
    Efficiently fetches Manhwa metadata using a cache-first approach.
    """
    base_url = "https://api.jikan.moe/v4/manga"
    headers = {'User-Agent': 'ManhwaProject/1.0'}
    
    # Optional limit for testing/sampling 
    subset = master_df.head(limit) if limit else master_df

    for index, row in subset.iterrows():
        original_title = row['title']
        filename = get_sanitized_filename(original_title)
        file_path = os.path.join(CACHE_DIR, filename)

        # 1. Check if data already exists (Efficiency Step)
        if os.path.exists(file_path):
            continue # Skip to next title

        print(f"[{index}] Fetching from API: {original_title}")
        
        try:
            # 2. API Request with Manhwa filter [cite: 6]
            params = {'q': original_title, 'type': 'manhwa', 'limit': 3}
            response = requests.get(base_url, params=params, headers=headers)
            
            if response.status_code == 200:
                results = response.json().get('data', [])
                
                # 3. Fuzzy Matching logic to ensure data integrity [cite: 10, 17]
                best_match = None
                highest_similarity = 0
                
                for item in results:
                    similarity = fuzz.token_sort_ratio(original_title, item['title'])
                    if similarity > highest_similarity:
                        highest_similarity = similarity
                        best_match = item
                
                # 4. Save only if match is confident (>80%)
                if best_match and highest_similarity > 80:
                    metadata = {
                        'original_title': original_title,
                        'mal_id': best_match['mal_id'],
                        'title_api': best_match['title'],
                        'rank': best_match.get('rank'),
                        'popularity': best_match.get('popularity'),
                        'score': best_match.get('score'),
                        'cover_url': best_match['images']['jpg']['large_image_url'],
                        'genres': [g['name'] for g in best_match.get('genres', [])],
                        'match_confidence': highest_similarity
                    }
                    
                    with open(file_path, 'w', encoding='utf-8') as f:
                        json.dump(metadata, f, indent=4)
                
                # 5. Respect Jikan Rate Limits (3 requests/sec)
                time.sleep(1.2) 
                
            elif response.status_code == 429:
                print("Rate limit hit. Cooling down for 30 seconds...")
                time.sleep(30)
                
        except Exception as e:
            print(f"Error processing {original_title}: {e}")
            continue
    print("finished")

# Run the smart loop on your master_list 
smart_enrichment_loop(master_list)

#### Remember that:
**master_list** = combination of two kaggle datasets

**json_cache** is a directory



This code find the number of manhwa's included in kaggle datasets but couldn't find with API:

In [36]:

def audit_json_cache(master_list, cache_dir='data/json_cache'):
    """
    Compares the master list against the JSON cache to identify missing data.
    """
    # 1. Get the list of all sanitized filenames currently in the cache
    cached_files = {f.replace('.json', '') for f in os.listdir(cache_dir) if f.endswith('.json')}
    
    # 2. Extract all titles from your master_list
    all_titles = master_list['title'].tolist()
    
    matched_titles = []
    missing_titles = []

    for title in all_titles:
        # We use the same sanitization logic used during the fetching process
        sanitized_name = "".join(x for x in title if x.isalnum()).lower()
        
        if sanitized_name in cached_files:
            matched_titles.append(title)
        else:
            missing_titles.append(title)

    # 3. Calculate statistics
    total = len(all_titles)
    found = len(matched_titles)
    missing = len(missing_titles)
    success_rate = (found / total) * 100

    print("--- DATASET AUDIT REPORT ---")
    print(f"Total Kaggle Titles:   {total}")
    print(f"Successfully Matched:  {found}")
    print(f"Missing from Cache:    {missing}")
    print(f"Coverage Success:      {success_rate:.2f}%")
    print("-" * 28)

    # Save the missing titles to a CSV for later inspection (e.g., the 'Let's Play' cases)
    missing_df = pd.DataFrame(missing_titles, columns=['missing_title'])
    missing_df.to_csv('data/missing_from_jikan.csv', index=False)
    print("List of missing titles saved to 'data/missing_from_jikan.csv'")
    
    return missing_titles

# Execute the audit
missing_manhwa = audit_json_cache(master_list)

--- DATASET AUDIT REPORT ---
Total Kaggle Titles:   1469
Successfully Matched:  723
Missing from Cache:    746
Coverage Success:      49.22%
----------------------------
List of missing titles saved to 'data/missing_from_jikan.csv'


### **Step 1: Create the Ultimate DataFrame**
First, we merge the Kaggle data with the Jikan metadata to have all URLs in one row.

In [38]:

def create_ultimate_research_df(master_list, cache_dir='data/json_cache'): 
    """
    Integrates Kaggle URLs and Jikan Metadata into a single research-ready DataFrame.
    """
    # Load Jikan metadata from JSON files
    json_list = []
    for filename in os.listdir(cache_dir):
        if filename.endswith('.json'):
            with open(os.path.join(cache_dir, filename), 'r', encoding='utf-8') as f:
                json_list.append(json.load(f))
    
    jikan_df = pd.DataFrame(json_list)
    
    # Inner Join: Only keep titles that exist in both Kaggle and Jikan Cache
    # We rename columns to be explicit about the timeline
    ultimate_df = pd.merge(
        master_list[['title', 'cover']], # 'cover' is the Kaggle 2022 URL
        jikan_df[['original_title', 'cover_url', 'score', 'rank', 'popularity']], 
        left_on='title', 
        right_on='original_title', 
        how='inner' #Inner Join only keeps rows where the title exists in 
        #both the Kaggle list and the Jikan Cache.
    )
    
    # Rename for clarity
    ultimate_df = ultimate_df.rename(columns={
        'cover': 'url_2022',
        'cover_url': 'url_2026'
    })
    
    # Drop redundant column
    if 'original_title' in ultimate_df.columns:
        ultimate_df.drop(columns=['original_title'], inplace=True)
        
    print(f"Ultimate DataFrame created with {len(ultimate_df)} titles.")
    return ultimate_df

# Execute
ultimate_df = create_ultimate_research_df(master_list)

Ultimate DataFrame created with 723 titles.


In [13]:
# Master listede olup Ultimate listede olmayanları bul
missing_8 = master_list[~master_list['title'].isin(ultimate_df['title'])]
print(missing_8['title'])

0                            Let's Play
3                   Midnight Poppy Land
7                            I Love Yoo
8                           Age Matters
10                       Fictional Skin
                     ...               
1464                  Found: His Family
1465    Pr&iacute;ncipe &amp; Caballero
1466                 Hold Back the Dark
1467                 Just For the Night
1468                  All Hallow's Even
Name: title, Length: 746, dtype: str


The ultimate_df consists of 723 manhwa/webtoon titles that are present in both the MyAnimeList (MAL) database and the two Kaggle datasets.

### **Step 2: The Dual-Version Image Downloader**
This function implements your logic: it tries to download the 2022 version (using headers) and the 2026 version. It tracks which ones were successful.

In [14]:
def download_dual_covers(df):
    """
    Downloads cover images for both 2022 (Kaggle) and 2026 (Jikan) versions.
    Tracks success status for data quality reporting.
    """
    base_dir = 'data/covers'
    dir_2022 = os.path.join(base_dir, 'v2022')
    dir_2026 = os.path.join(base_dir, 'v2026')
    
    os.makedirs(dir_2022, exist_ok=True)
    os.makedirs(dir_2026, exist_ok=True)
    
    # Essential headers to bypass server-side blocks
    headers = {
        'Referer': 'https://www.webtoons.com/',
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }

    # Tracking lists to update the DataFrame later
    status_2022 = []
    status_2026 = []

    for index, row in df.iterrows():
        title = row['title']
        # Create a safe filename
        safe_name = "".join(x for x in title if x.isalnum()).lower() + ".jpg"
        
        path_2022 = os.path.join(dir_2022, safe_name)
        path_2026 = os.path.join(dir_2026, safe_name)

        # 1. Process 2022 Image (Kaggle)
        success_22 = False
        if os.path.exists(path_2022):
            success_22 = True
        elif pd.notna(row['url_2022']):
            try:
                resp = requests.get(row['url_2022'], headers=headers, timeout=10)
                if resp.status_code == 200:
                    with open(path_2022, 'wb') as f:
                        f.write(resp.content)
                    success_22 = True
            except:
                pass
        
        # 2. Process 2026 Image (Jikan)
        success_26 = False
        if os.path.exists(path_2026):
            success_26 = True
        elif pd.notna(row['url_2026']):
            try:
                resp = requests.get(row['url_2026'], headers=headers, timeout=10)
                if resp.status_code == 200:
                    with open(path_2026, 'wb') as f:
                        f.write(resp.content)
                    success_26 = True
            except:
                pass

        status_2022.append(success_22)
        status_2026.append(success_26)
        
        if index % 50 == 0:
            print(f"Progress: {index}/{len(df)} titles processed.")
        
        # Respect servers to avoid getting banned
        time.sleep(0.2)

    # Add status flags to the DataFrame
    df['has_v2022'] = status_2022
    df['has_v2026'] = status_2026
    
    return df

# Start the download process
ultimate_df = download_dual_covers(ultimate_df)

Progress: 0/723 titles processed.
Progress: 50/723 titles processed.
Progress: 100/723 titles processed.
Progress: 150/723 titles processed.
Progress: 200/723 titles processed.
Progress: 250/723 titles processed.
Progress: 300/723 titles processed.
Progress: 350/723 titles processed.
Progress: 400/723 titles processed.
Progress: 450/723 titles processed.
Progress: 500/723 titles processed.
Progress: 550/723 titles processed.
Progress: 600/723 titles processed.
Progress: 650/723 titles processed.
Progress: 700/723 titles processed.


Run this code after the downloader finishes. It will tell you exactly what is missing and why.

In [19]:
# 1. Filter the rows where the Kaggle (2022) download failed
broken_2022_df = ultimate_df[ultimate_df['has_v2022'] == False]

# 2. Filter the rows where the Jikan (2026) download failed
broken_2026_df = ultimate_df[ultimate_df['has_v2026'] == False]

# 3. Print the results
print("--- BROKEN URL REPORT ---")
print(f"Total Titles Checked: {len(ultimate_df)}")
print(f"Broken Kaggle (2022) URLs: {len(broken_2022_df)}")
print(f"Broken Jikan (2026) URLs:  {len(broken_2026_df)}")
print("-" * 25)

# 4. Display the list of broken Kaggle titles
if not broken_2022_df.empty:
    print("\nSample of Titles with Broken 2022 Links:")
    print(broken_2022_df[['title', 'url_2022']].head(10))
    
    # Optional: Save the broken list to a CSV for your report
    broken_2022_df[['title', 'url_2022']].to_csv('data/broken_links_2022.csv', index=False)

--- BROKEN URL REPORT ---
Total Titles Checked: 723
Broken Kaggle (2022) URLs: 0
Broken Jikan (2026) URLs:  0
-------------------------


In [20]:
# Group A: Perfect matches (Before & After comparison possible)
comparable_df = ultimate_df[(ultimate_df['has_v2022'] == True) & (ultimate_df['has_v2026'] == True)]

# Group B: Current only (Only 2026 analysis possible)
current_only_df = ultimate_df[(ultimate_df['has_v2022'] == False) & (ultimate_df['has_v2026'] == True)]

print(f"Titles available for Before/After Analysis: {len(comparable_df)}")
print(f"Titles available for Current Popularity Analysis: {len(ultimate_df[ultimate_df['has_v2026'] == True])}")

Titles available for Before/After Analysis: 723
Titles available for Current Popularity Analysis: 723


**comparable_df** = manhwa's which urls coming from jikan and kaggle datasets works.

In [22]:
# Create the final clean dataset for OpenCV and Statistics
# Keep only titles where both versions were successfully downloaded
final_analysis_df = ultimate_df[(ultimate_df['has_v2022'] == True) & (ultimate_df['has_v2026'] == True)].copy()

print(f"Final sample size for research: {len(final_analysis_df)}")
final_analysis_df.to_csv('data/final_manhwa_research_data.csv', index=False)

Final sample size for research: 723


##### **For later purposes:** Another dataset with top 200 manhwas from MyAnimeList using Jikan API

In [21]:
def fetch_top_manhwa_standalone(limit=200):
    """
    Fetches the current Top 200 Manhwa from Jikan directly,
    independent of the Kaggle dataset.
    """
    base_url = "https://api.jikan.moe/v4/top/manga"
    top_manhwas = []
    
    # Each page gives 25 results, so we need 8 pages for 200 results
    for page in range(1, 9):
        params = {'type': 'manhwa', 'page': page}
        response = requests.get(base_url, params=params)
        if response.status_code == 200:
            data = response.json().get('data', [])
            for item in data:
                top_manhwas.append({
                    'title': item['title'],
                    'rank': item['rank'],
                    'popularity': item['popularity'],
                    'score': item['score'],
                    'cover_url': item['images']['jpg']['large_image_url']
                })
        print(f"Fetched page {page}...")
        time.sleep(1.2) # API Rate Limit
        
    return pd.DataFrame(top_manhwas)

# Execute
top_200_df = fetch_top_manhwa_standalone()

Fetched page 1...
Fetched page 2...
Fetched page 3...
Fetched page 4...
Fetched page 5...
Fetched page 6...
Fetched page 7...
Fetched page 8...


In [23]:
# 1. Create a dedicated directory and subdirectories for the Top 200 Research
# Structure: data/top_200_research/json/ and data/top_200_research/covers/
json_path = 'data/top_200_research/json'
covers_path = 'data/top_200_research/covers'

os.makedirs(json_path, exist_ok=True)
os.makedirs(covers_path, exist_ok=True)

# 2. Save the Top 200 DataFrame to the new dedicated JSON directory
top_200_df.to_json(f'{json_path}/top_200_metadata.json', orient='records', indent=4)

print(f"Top 200 metadata saved to: {json_path}/top_200_metadata.json")
print(f"Covers will be downloaded to: {covers_path}/")

Top 200 metadata saved to: data/top_200_research/json/top_200_metadata.json
Covers will be downloaded to: data/top_200_research/covers/


In [24]:
def download_top_200_covers(df, save_folder='data/top_200_research/covers'):
    """
    Downloads cover images for the Top 200 dataset into a specific folder.
    """
    # make directory or check it
    os.makedirs(save_folder, exist_ok=True)
    
    print(f"🚀 Starting download of {len(df)} covers to: {save_folder}...")
    
  
    #adding a user agent so that server won't see us as a bot
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    
    downloaded_count = 0
    
    for index, row in df.iterrows():
        # make file name safe
        safe_name = "".join(x for x in row['title'] if x.isalnum() or x in "._- ").strip().replace(" ", "_").lower() + ".jpg"
        file_path = os.path.join(save_folder, safe_name)
        
        # If cover already downoaded, dont download it again
        if os.path.exists(file_path):
            continue
            
        try:
            response = requests.get(row['cover_url'], headers=headers, timeout=15)
            if response.status_code == 200:
                with open(file_path, 'wb') as f:
                    f.write(response.content)
                downloaded_count += 1
                
                # Her 20 resimde bir durum güncellemesi yapalım
                if downloaded_count % 25 == 0:
                    print(f"✅ Progress: {downloaded_count} images downloaded...")
            
            # a short wait in order not to not catch by API restrictions
            time.sleep(0.5)
            
        except Exception as e:
            print(f"❌ Error downloading {row['title']}: {e}")



download_top_200_covers(top_200_df)

print("\n✨ Download process finished! You can now check the 'data/top_200_research/covers' folder.")

🚀 Starting download of 200 covers to: data/top_200_research/covers...

✨ Download process finished! You can now check the 'data/top_200_research/covers' folder.


## Analysis of art styles using OpenCV

### Phase 1: Feature Selection (The "Aesthetic" Metrics)
Engineer 5 specific numerical features that represent a cover's art style.

**brightness_mean**: Extracted from the Value (V) channel in HSV. Represents if the cover is generally dark or light.

**saturation_mean** : Extracted from the Saturation (S) channel in HSV. Represents how vibrant or dull the colors are.

**contrast_rms**: Root Mean Square (RMS) contrast of the grayscale image. Measures the difference between light and dark areas.

**edge_density**: Using Canny Edge Detection. Represents the complexity, detail, or "clutter" of the line art.

**color_entropy**: Calculates the randomness of the color distribution. A high entropy means a very complex color palette; low entropy means a minimalist or monochromatic palette.

### Phase 2: Creating Automated Extraction and Change Detection Functions
Create two Python functions: the first should accept an image path, calculate the five specified metrics, and return them as a dictionary. The second should compare the 2022 and 2026 cover data to detect if the cover has changed.

In [25]:
def extract_visual_features(image_path):
    """
    Reads an image and extracts 5 key aesthetic features for Data Science analysis.
    Returns a dictionary of features or None if the image is unreadable.
    """
    try:
        # 1. Load Image
        img_bgr = cv2.imread(image_path)
        if img_bgr is None:
            return None
        
        # 2. Convert Color Spaces
        img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
        
        features = {}
        
        # FEATURE 1 & 2: Brightness and Saturation (from HSV)
        # HSV -> Hue (0), Saturation (1), Value/Brightness (2)
        features['saturation_mean'] = np.mean(img_hsv[:, :, 1])
        features['brightness_mean'] = np.mean(img_hsv[:, :, 2])
        
        # FEATURE 3: RMS Contrast
        # Standard deviation of the pixel intensities in grayscale
        features['contrast_rms'] = np.std(img_gray)
        
        # FEATURE 4: Edge Density (Complexity)
        # Apply slight blur to remove noise, then Canny Edge
        blurred = cv2.GaussianBlur(img_gray, (3, 3), 0)
        edges = cv2.Canny(blurred, 100, 200)
        total_pixels = img_gray.shape[0] * img_gray.shape[1]
        features['edge_density'] = (np.sum(edges > 0) / total_pixels) * 100
        
        # FEATURE 5: Color Entropy (Palette Complexity)
        # Calculate histogram of grayscale, normalize it, and find entropy
        hist = cv2.calcHist([img_gray], [0], None, [256], [0, 256])
        hist_normalized = hist.ravel() / hist.sum()
        features['color_entropy'] = entropy(hist_normalized, base=2)
        
        # Round all values for a cleaner DataFrame
        return {k: round(v, 4) for k, v in features.items()}
        
    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return None

In [26]:
# ---- This is a function for detecting cover changes between years ---- 
# --- IMPROVED CHANGE DETECTION LOGIC ---

def detect_cover_changes(df):
    """
    Identifies if a cover has changed by looking at multiple visual dimensions.
    If any key feature shifts significantly, we flag it as 'changed'.
    """
    
    # 1. Define sensitivity thresholds for each metric
    # These are percentages or absolute shifts based on standard deviations
    thresholds = {
        'edge': 0.8,       # Significant shift in line complexity
        'sat': 5.0,        # Significant shift in color vibrancy
        'bright': 5.0,     # Significant shift in lighting/darkness
        'entropy': 0.1     # Significant shift in palette complexity
    }

    # 2. Calculate absolute differences for all metrics
    df['diff_edge'] = (df['v2022_edge_density'] - df['v2026_edge_density']).abs()
    df['diff_sat'] = (df['v2022_saturation_mean'] - df['v2026_saturation_mean']).abs()
    df['diff_bright'] = (df['v2022_brightness_mean'] - df['v2026_brightness_mean']).abs()
    df['diff_entropy'] = (df['v2022_color_entropy'] - df['v2026_color_entropy']).abs()

    # 3. Combined Change Flag
    # If ANY of the metrics cross their threshold, mark as TRUE
    df['cover_changed'] = (
        (df['diff_edge'] > thresholds['edge']) | 
        (df['diff_sat'] > thresholds['sat']) | 
        (df['diff_bright'] > thresholds['bright']) | 
        (df['diff_entropy'] > thresholds['entropy'])
    )
    
    return df

### Phase 3: Integration & Master Table Creation
We will iterate through final_manhwa_research_data, apply the function to each cover, and merge these new numerical features into the main dataframe.

In [27]:
# --- MULTI-YEAR VISUAL ANALYSIS ---

def batch_extract_features(df, folder_path, year_label):
    """
    Helper function to extract features for all images in a specific folder.
    """
    results = []
    print(f"🔄 Extracting visual features for: {folder_path}...")
    
    for index, row in df.iterrows():
        # Build file path
        # Example: "True Beauty" -> "truebeauty.jpg"
        clean_title = "".join(x for x in row['title'] if x.isalnum()).lower()
        safe_name = f"{clean_title}.jpg"

        file_path = os.path.join(folder_path, safe_name)
        
        # Run Extraction
        features = extract_visual_features(file_path)
        
        if features:
            # Add year_label to keys (e.g., v2022_edge_density)
            features = {f"{year_label}_{k}": v for k, v in features.items()}
            results.append(features)
        else:
            # Handle missing files with labeled NaNs
            results.append({f"{year_label}_{k}": np.nan for k in ['saturation_mean', 'brightness_mean', 'contrast_rms', 'edge_density', 'color_entropy']})
            
    return pd.DataFrame(results)

# 1. Extract 2022 Features
visual_2022_df = batch_extract_features(final_analysis_df, 'data/covers/v2022', 'v2022')

# 2. Extract 2026 Features
visual_2026_df = batch_extract_features(final_analysis_df, 'data/covers/v2026', 'v2026')

# 3. Combine everything into the Ultimate Dataset
manhwa_art_evolution_2022_2026 = pd.concat([
    final_analysis_df.reset_index(drop=True), 
    visual_2022_df, 
    visual_2026_df
], axis=1)

# 4. Cover Change Detection
# Apply the multi-metric detection logic
manhwa_art_evolution_2022_2026 = detect_cover_changes(manhwa_art_evolution_2022_2026)

# Split into subsets for easier plotting later
df_changed = manhwa_art_evolution_2022_2026[manhwa_art_evolution_2022_2026['cover_changed'] == True].copy()
df_unchanged = manhwa_art_evolution_2022_2026[manhwa_art_evolution_2022_2026['cover_changed'] == False].copy()

# 5. Save the final multi-year research file 
# You can keep your original name or use the one you prefer here
output_path = 'data/final_manhwa_research_data.csv' 
manhwa_art_evolution_2022_2026.to_csv(output_path, index=False)

print(f"Multi-year analysis complete!")
print(f"Dataset updated and saved to: {output_path}")
print(f"INFO: {len(df_changed)} covers identified as changed.")
print(f"INFO: {len(df_unchanged)} covers identified as unchanged.")

🔄 Extracting visual features for: data/covers/v2022...
🔄 Extracting visual features for: data/covers/v2026...
Multi-year analysis complete!
Dataset updated and saved to: data/final_manhwa_research_data.csv
INFO: 714 covers identified as changed.
INFO: 9 covers identified as unchanged.


In [28]:
# Create a list of titles that failed the analysis
missing_titles = manhwa_art_evolution_2022_2026[manhwa_art_evolution_2022_2026['v2022_edge_density'].isna()]['title'].tolist()
print(f"Number of failed images: {len(missing_titles)}")

Number of failed images: 0
